In [2]:
#import libraries here
# --- 1) Import + reload (important when you edit the module) ---
import importlib
import re_lib.eng_var as ev


importlib.reload(ev)

<module 're_lib.eng_var' from 'C:\\Users\\dgolovichev\\Documents\\SynologyDrive\\Vibe_Coding\\Python_for_Calcs\\re_lib\\eng_var.py'>

## General Information

Introdactory text with figures as appropriate  

*Italic text* or _Italic text_
**Bold text** or __Bold text__


List: 
* Item 1
* Item 2

Numberred List:
1. Item 1
2. Item 2

[Link](https://jupyter.org)

`inline code`

***


Inline: $y = a + bx$ renders as \(y=a+bx\).

Display: $$P(A|B) = \frac{P(B|A)P(A)}{P(B)}$$ renders as a centered equation.

Table 1. 
| Header 1 | Header 2 |
| :--- | :--- |
| Cell A | Cell B |
| Cell C | Cell D |

### Calcs Example (Engineering Workflow)
This example shows a practical structural workflow:
1. Define equations first (`v.eq.*`) so formulas are reusable and report-ready.
2. Define variables with units/comments.
3. Display symbolic, numeric, and full formatted equations.
4. Control numbering/alignment and extract plain numeric values for downstream checks.


In [3]:
# --- 2) Create environment ---
v = ev.EngEnv(
    eq_numbers=True,
    center_equations=True,
    output='notebook',
    notebook_render_mode='html',   # use 'latex' for pure MathJax mode
)

# --- 3) Define reusable equations first ---
v.eq.a = '(A_s*f_y)/(0.85*b*f_c)'
v.eq.phiM_n = 'phi*(A_s*f_y*(d-a/2))'
v.eq.DCR = 'M_u/phiM_n'


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [328]:
# --- 4) Define variables (with units + comments) ---
v.phi = (0.90, '', 'Strength reduction factor')
v.A_s = (4.00, 'in^2', 'Tension steel area')
v.f_y = (60.0, 'ksi', 'Steel yield strength')
v.b   = (12.0, 'in',  'Section width')
v.f_c = (4.00, 'ksi', 'Concrete compressive strength')
v.d   = (22.0, 'in',  'Effective depth')
v.M_u = (250.0, 'kip*ft', 'Factored moment demand')

# show a single variable on demand
v.A_s.show()


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<Quantity(340.0, 'kip')>

In [324]:
# --- 5) Show equation formatting modes ---
# symbolic only
v.eq.a.show(view='sym', number=True, center=False)

# numeric substitution + result
v.eq.a.show(view='num', out_units='in', fmt='.4f', number=True, center=False)

# full: symbolic + substitution + result
v.eq.phiM_n.show(view='full', out_units='kip*ft', fmt='.3f', number=True, center=False)


340.0 kip
340.0 kip
340.0


In [325]:
# --- 6) Default behavior: symbolic before variables, numeric after ---
tmp = ev.EngEnv(auto_display=False, eq_numbers=True)
tmp.eq.M_n = 'F_y*Z_x'

# No variables yet -> defaults to symbolic
tmp.eq.M_n.show(number=True)

tmp.F_y = (50, 'ksi')
tmp.Z_x = (100, 'in^3')

# Now all inputs exist -> defaults to numeric
tmp.eq.M_n.show(out_units='kip*in', number=True)


<IPython.core.display.Math object>

<Quantity(1.7, 'kip')>

In [318]:
# --- 7) Numbering + labels ---
v.reset_eq(0)

v.eq.a.show(view='sym', number=True, tag='@eq-a')
v.eq.phiM_n.show(view='sym', number=True, tag='@eq-phiMn')

n_a = v.eq_label('eq-a')
n_phi = v.eq_label('eq-phiMn')
v.render_equation(rf"\text{{Use Eq. ({n_a}) in Eq. ({n_phi})}}", center=False)


$$
P_D = \frac{0.85 \cdot A_{c} \cdot f_{c}}{2} = \frac{0.85 \cdot 200.00\,\mathrm{in^2} \cdot 4.00\,\mathrm{ksi}}{2} = 340.00\,\mathrm{kip}\qquad\text{(1)}
$$

$$
F_e = \pi^{2} \cdot E \cdot \left(\frac{K \cdot L}{r}\right)^{- 2} = \pi^{2} \cdot 29000.000\,\mathrm{ksi} \cdot \left(\frac{1.000 \cdot 216.000\,\mathrm{in}}{2.100\,\mathrm{in}}\right)^{- 2} = 27.054\,\mathrm{ksi}\qquad\text{(2)}
$$



In [319]:
# --- 8) Value-only outputs for design logic / AI workflows ---
a_in = v.eq.a.value('in')
phiMn_kft = v.eq.phiM_n.value('kip*ft')
dcr = v.eq.DCR.value()

print(f'a = {a_in:.4f} in')
print(f'phiM_n = {phiMn_kft:.3f} kip-ft')
print(f'DCR = {dcr:.3f}')


$$
\begin{aligned}
& \text{Using Eq. (2)}
\end{aligned}
$$

$$
\begin{aligned}
& P_D = \frac{0.85 \cdot 200.00\,\mathrm{in^2} \cdot 4.00\,\mathrm{ksi}}{2} = 340.00\,\mathrm{kip}\qquad\text{(3)}
\end{aligned}
$$

$$
P_D = \frac{0.85 \cdot 200.00\,\mathrm{in^2} \cdot 4.00\,\mathrm{ksi}}{2} = 340.00\,\mathrm{kip}\qquad\text{(4)}
$$



<Quantity(340.0, 'kip')>

In [320]:
# --- 9) Override inputs without mutating base equation ---
phiMn_hi_fc = v.eq.phiM_n(f_c=5.0*v.ksi).show(view='num', out_units='kip*ft', fmt='.3f', number=True)
phiMn_lo_As = v.eq.phiM_n(A_s=3.2*v.inch**2).show(view='num', out_units='kip*ft', fmt='.3f', number=True)

# left align for report style
v.eq.DCR.show(view='num', center=False, number=True)


$$
\begin{aligned}
& f_c = 4\,\mathrm{ksi}\;\;\text{(Concrete compressive strength)}
\end{aligned}
$$

$$
\begin{aligned}
& A_c = 200\,\mathrm{in^2}\;\;\text{(Concrete area)}
\end{aligned}
$$

$$
P_D = \frac{0.85 \cdot A_{c} \cdot f_{c}}{2} = \frac{0.85 \cdot 200.00\,\mathrm{in^2} \cdot 4.00\,\mathrm{ksi}}{2} = 340.00\,\mathrm{kip}\qquad\text{(5)}
$$



<Quantity(340.0, 'kip')>

In [321]:
# --- 10) Unicode / Greek-friendly names ---
v.eq.define('ΔP', 'rho*g*h')
v.rho = (62.4, 'lbf/ft^3', 'Fluid unit weight')
v.g = (1.0, '', 'Normalized gravity factor')
v.h = (10.0, 'ft', 'Fluid head')

# Symbolic output with unicode equation name
v.eq['ΔP'].show(view='sym', number=True, center=False)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

(True, 0.9, 0.7287379972565158, <Quantity(343.058824, 'kip * foot')>)